In [2]:
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [4]:
from langchain_groq import ChatGroq
model=ChatGroq(model="Gemma2-9b-It")
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000019405722110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019405723580>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage
messages=[
    HumanMessage(content="Hello How are you?")
]
response=model.invoke(messages)
response

AIMessage(content="As an AI, I don't have feelings, but I'm here and ready to help! How can I assist you today? 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 14, 'total_tokens': 46, 'completion_time': 0.058181818, 'prompt_time': 0.001902795, 'queue_time': 0.233185554, 'total_time': 0.060084613}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-e767924f-cde1-428b-b92c-fd0a93e2bd35-0', usage_metadata={'input_tokens': 14, 'output_tokens': 32, 'total_tokens': 46})

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Abhinav")],
    config=config
)

In [10]:
response.content

"Hi Abhinav, it's nice to meet you! 👋\n\nWhat can I do for you today?\n"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Abhinav")],
    config=config
)

In [12]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Abhinav!  \n\nI remember that from our earlier introduction. 😊  How can I help you further?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 102, 'total_tokens': 131, 'completion_time': 0.052727273, 'prompt_time': 0.010237437, 'queue_time': 0.23599244100000002, 'total_time': 0.06296471}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-4d5ed419-e559-47f4-a247-730640548dfb-0', usage_metadata={'input_tokens': 102, 'output_tokens': 29, 'total_tokens': 131})

In [13]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to use it! 😊\n"

In [14]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Hey John! Nice to meet you. 👋\n\nWhat can I do for you today? 😊  \n'

In [15]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John!  😄  \n\nI remember now.  \n\n\n'

In [37]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        ("user","{input}")
    ]
)

chain=prompt|model

In [38]:
chain.invoke({"input":"My name is abhinav"})

AIMessage(content="Hello Abhinav!  It's nice to meet you.  \n\nI'm here to help with any questions you have to the best of my ability.  \n\nWhat can I do for you today? 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 32, 'total_tokens': 82, 'completion_time': 0.090909091, 'prompt_time': 0.002359476, 'queue_time': 0.230908963, 'total_time': 0.093268567}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-d95d564d-736f-4cb3-968f-75ab4d0a8eca-0', usage_metadata={'input_tokens': 32, 'output_tokens': 50, 'total_tokens': 82})

In [39]:
config={"configurable":{"session_id":"chat2"}}
with_message_history=RunnableWithMessageHistory(chain,get_session_history)
response=with_message_history.invoke(
    {"input":"My name is abhinav"},
    config=config
)

response.content

'Your name is Abhinav.  \n\nI remember! 😊  Is there anything else I can help you with? \n'

In [40]:
response=with_message_history.invoke(
    {"input":"what is my name?"},
    config=config
)

response.content

'Your name is Abhinav.  \n'

In [41]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}."
        ),
        ("user","{messages}")
    ]
)

chain = prompt | model

In [42]:
config={"configurable":{"session_id":"chat3"}}
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
response=with_message_history.invoke(
    {"messages":"My name is abhinav","language":"Hindi"},
    config=config
)

response.content

'नमस्ते अभिनव!  \n\nमैं आपकी मदद करने के लिए यहाँ हूँ।  आपके कोई सवाल हैं तो बेझिझक पूछिए, मैं अपनी पूरी कोशिश करूँगा कि आपको सबसे अच्छा जवाब दूँ। 😊  \n'

In [43]:
response=with_message_history.invoke(
    {"messages":"What is my name?","language":"Hindi"},
    config=config
)

response.content

'आपका नाम अभिनव है। 😊 \n'

In [45]:
from langchain_core.messages import SystemMessage,trim_messages,AIMessage
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

e:\GENAI\LANGCHAIN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\GENAI\LANGCHAIN\venv\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://doc

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [46]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As an AI, I don't have personal preferences like liking ice cream flavors.  \n\nCould you tell me what flavors you enjoy? Maybe I can suggest some new ones you'd like to try! 😊 \n"

In [47]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [48]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I'm not able to access past conversations. \n\nTo answer your question, I need to know your name! 😊  What is it? \n"